In [ ]:
import cv2
import numpy as np
import glob

chessboard_size = (9,6)

# Prepare object points
objp = np.zeros((chessboard_size[0] * chessboard_size[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:chessboard_size[0], 0:chessboard_size[1]].T.reshape(-1, 2)

# Storage arrays
objpoints = []  # 3D points
imgpoints = []  # 2D points

# Load calibration images
images = glob.glob(r"D:\Cv_img_segmentation_pipeline\Calibration\images\*.jpeg")

# Detect corners

img_size = None

for img_path in images:
    
    img = cv2.imread(img_path)
    gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    if img_size is None:
        img_size = gray.shape[::-1]

    ret, corners = cv2.findChessboardCorners(gray, chessboard_size, None)

    if ret:
        objpoints.append(objp)
        imgpoints.append(corners)

        cv2.drawChessboardCorners(img, chessboard_size, corners, ret)
        cv2.imshow("Corners", img)
        cv2.waitKey(200)

cv2.destroyAllWindows()

print("Total images found:", len(images))
print("Valid detections:", len(objpoints))
print("Valid imgpoints:", len(imgpoints))


# Run calibration
ret, K, dist, rvecs, tvecs = cv2.calibrateCamera(
    objpoints, imgpoints, img_size, None, None
)


# Save Results
np.save("camera_matrix.npy", K)
np.save("dist_coeffs.npy", dist)

print("Camera Matrix:\n", K)
print("Distortion:\n", dist)


# Validation
img = cv2.imread(images[0])
h, w = img.shape[:2]

new_K, roi = cv2.getOptimalNewCameraMatrix(K, dist, (w, h), 1, (w, h))
undistorted = cv2.undistort(img, K, dist, None, new_K)

cv2.imshow("Original", img)
cv2.imshow("Undistorted", undistorted)
cv2.waitKey(0)
cv2.destroyAllWindows()

Total images found: 21
Valid detections: 20
Valid imgpoints: 20
Camera Matrix:
 [[1.23218523e+03 0.00000000e+00 7.98707648e+02]
 [0.00000000e+00 1.23224786e+03 4.59267226e+02]
 [0.00000000e+00 0.00000000e+00 1.00000000e+00]]
Distortion:
 [[ 0.02001483  0.09007609  0.00119459  0.00396159 -0.5743593 ]]


: 